In [ ]:
import math
from pathlib import Path
from typing import Iterable, List, Sequence, Tuple, Union

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


DATA_PATH = Path("airports.csv")
OUT_DIR = Path("outputs")



def is_us_airport(lat: float, lon: float) -> bool:
#check if airport is us
    if pd.isna(lat) or pd.isna(lon):
        return False

    contiguous = (24.0 <= lat <= 50.5) and (-125.0 <= lon <= -66.0)

    # Alaska
    alaska = (51.0 <= lat <= 72.5) and (-170.0 <= lon <= -130.0)

    # Hawaii
    hawaii = (18.0 <= lat <= 23.5) and (-161.0 <= lon <= -154.0)

    return contiguous or alaska or hawaii


def get_jfk(df_airports: pd.DataFrame) -> pd.Series:
#Return the row for JFK from the airports table
    jfk = df_airports.loc[df_airports["faa"].astype(str).str.upper() == "JFK"]
    if jfk.empty:
        raise ValueError("Could not find JFK in airports.csv (faa == 'JFK').")
    return jfk.iloc[0]


def ensure_out_dir() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
# <- check that output dir exist lol

def save_html(fig: go.Figure, name: str) -> Path:
#Save Plotly figure to HTML
    ensure_out_dir()
    out_path = OUT_DIR / f"{name}.html"
    fig.write_html(out_path, include_plotlyjs="cdn")
    return out_path



def plot_all_airports_world(df: pd.DataFrame, color_by_altitude: bool = False) -> go.Figure:
#World map with points for all airports
    color = "alt" if color_by_altitude else None
    fig = px.scatter_geo(
        df,
        lat="lat",
        lon="lon",
        hover_name="name",
        hover_data={"faa": True, "alt": True, "tzone": True, "lat": False, "lon": False},
        color=color,
        title="All airports (world map)" + (" — color coded by altitude" if color_by_altitude else ""),
    )
    fig.update_geos(showland=True)
    fig.update_layout(margin=dict(l=10, r=10, t=60, b=10))
    return fig


def plot_airports_us_and_outside(df: pd.DataFrame, color_by_altitude: bool = False) -> Tuple[go.Figure, go.Figure]:
    #Identify airports outside the US
     # Map of outside-US airports world map
      # Map of US-only airports
    df = df.copy()
    df["in_us"] = df.apply(lambda r: is_us_airport(r["lat"], r["lon"]), axis=1)

    df_us = df[df["in_us"]].copy()
    df_out = df[~df["in_us"]].copy()

    color = "alt" if color_by_altitude else None

    fig_out = px.scatter_geo(
        df_out,
        lat="lat",
        lon="lon",
        hover_name="name",
        hover_data={"faa": True, "alt": True, "tzone": True, "lat": False, "lon": False},
        color=color,
        title="Airports outside the US (heuristic from lat/lon)" + (" — color coded by altitude" if color_by_altitude else ""),
    )
    fig_out.update_geos(showland=True)
    fig_out.update_layout(margin=dict(l=10, r=10, t=60, b=10))

    fig_us = px.scatter_geo(
        df_us,
        lat="lat",
        lon="lon",
        hover_name="name",
        hover_data={"faa": True, "alt": True, "tzone": True, "lat": False, "lon": False},
        color=color,
        scope="usa",
        title="Airports in the US (heuristic from lat/lon)" + (" — color coded by altitude" if color_by_altitude else ""),
    )
    fig_us.update_layout(margin=dict(l=10, r=10, t=60, b=10))

    return fig_out, fig_us


def _route_lines_trace(
    origin_lat: float, origin_lon: float, dest_lats: Sequence[float], dest_lons: Sequence[float]
) -> go.Scattergeo:

    #Create a single Scattergeo trace containing multiple independent line segments.
    #Plotly separates segments with None.

    lats: List[Union[float, None]] = []
    lons: List[Union[float, None]] = []
    for lat, lon in zip(dest_lats, dest_lons):
        lats.extend([origin_lat, lat, None])
        lons.extend([origin_lon, lon, None])

    return go.Scattergeo(
        lat=lats,
        lon=lons,
        mode="lines",
        line=dict(width=2),
        hoverinfo="skip",
        name="Routes",
    )


def plot_route_from_nyc(df: pd.DataFrame, faa: str, us_only_if_us: bool = True) -> go.Figure:

    #function that takes an FAA abbreviation and plots a world map and a line from NYC (JFK) to that airport

    #Note: If the airport is in the US, make a US-only map (scope='usa') when us_only_if_us=True.

    df = df.copy()
    faa = str(faa).upper()

    jfk = get_jfk(df)
    target = df.loc[df["faa"].astype(str).str.upper() == faa]
    if target.empty:
        raise ValueError(f"FAA code '{faa}' not found in airports.csv.")
    target = target.iloc[0]

    in_us = is_us_airport(float(target["lat"]), float(target["lon"]))
    use_usa_scope = bool(us_only_if_us and in_us)

    if use_usa_scope:
        base = px.scatter_geo(
            df,
            lat="lat",
            lon="lon",
            hover_name="name",
            hover_data={"faa": True, "alt": True, "tzone": True, "lat": False, "lon": False},
            scope="usa",
            title=f"Route from NYC (JFK) to {faa} — US-only scope",
        )
    else:
        base = px.scatter_geo(
            df,
            lat="lat",
            lon="lon",
            hover_name="name",
            hover_data={"faa": True, "alt": True, "tzone": True, "lat": False, "lon": False},
            title=f"Route from NYC (JFK) to {faa} — world scope",
        )
        base.update_geos(showland=True)

    # Routeline
    line = _route_lines_trace(float(jfk["lat"]), float(jfk["lon"]), [float(target["lat"])], [float(target["lon"])])
    base.add_trace(line)

    # points
    base.add_trace(
        go.Scattergeo(
            lat=[float(jfk["lat"])],
            lon=[float(jfk["lon"])],
            mode="markers",
            marker=dict(size=10),
            name="JFK",
            text=["JFK (NYC)"],
            hoverinfo="text",
        )
    )
    base.add_trace(
        go.Scattergeo(
            lat=[float(target["lat"])],
            lon=[float(target["lon"])],
            mode="markers",
            marker=dict(size=10),
            name=faa,
            text=[f"{faa}: {target['name']}"],
            hoverinfo="text",
        )
    )

    base.update_layout(margin=dict(l=10, r=10, t=60, b=10))
    return base


def plot_routes_from_nyc(df: pd.DataFrame, faas: Sequence[str], us_only_if_us: bool = False) -> go.Figure:

    df = df.copy()
    faas = [str(x).upper() for x in faas]

    jfk = get_jfk(df)
    targets = df[df["faa"].astype(str).str.upper().isin(faas)].copy()

    missing = sorted(set(faas) - set(targets["faa"].astype(str).str.upper()))
    if missing:
        raise ValueError(f"FAA codes not found in airports.csv: {missing}")

    targets["in_us"] = targets.apply(lambda r: is_us_airport(r["lat"], r["lon"]), axis=1)

    use_usa_scope = bool(us_only_if_us and targets["in_us"].all())

    if use_usa_scope:
        fig = px.scatter_geo(
            df,
            lat="lat",
            lon="lon",
            hover_name="name",
            hover_data={"faa": True, "alt": True, "tzone": True, "lat": False, "lon": False},
            scope="usa",
            title=f"Routes from NYC (JFK) to {len(faas)} airports — US-only scope",
        )
    else:
        fig = px.scatter_geo(
            df,
            lat="lat",
            lon="lon",
            hover_name="name",
            hover_data={"faa": True, "alt": True, "tzone": True, "lat": False, "lon": False},
            title=f"Routes from NYC (JFK) to {len(faas)} airports — world scope",
        )
        fig.update_geos(showland=True)

    #multi-segment line trace
    line = _route_lines_trace(float(jfk["lat"]), float(jfk["lon"]), targets["lat"].astype(float), targets["lon"].astype(float))
    fig.add_trace(line)

    #JFK marker
    fig.add_trace(
        go.Scattergeo(
            lat=[float(jfk["lat"])],
            lon=[float(jfk["lon"])],
            mode="markers",
            marker=dict(size=10),
            name="JFK",
            text=["JFK (NYC)"],
            hoverinfo="text",
        )
    )

    fig.update_layout(margin=dict(l=10, r=10, t=60, b=10))
    return fig


def euclidean_distance_deg(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    return math.sqrt((lat2 - lat1) ** 2 + (lon2 - lon1) ** 2)


def geodesic_distance_km(lat1: float, lon1: float, lat2: float, lon2: float, R_km: float = 6371.0) -> float:

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    lam1 = math.radians(lon1)
    lam2 = math.radians(lon2)

    dphi = phi2 - phi1
    dlam = lam2 - lam1
    phi_m = 0.5 * (phi1 + phi2)

    term1 = 2.0 * math.sin(dphi / 2.0) * math.cos(dlam / 2.0)
    term2 = 2.0 * math.cos(phi_m) * math.sin(dlam / 2.0)

    return R_km * math.sqrt(term1 * term1 + term2 * term2)


def plot_distance_distributions(df: pd.DataFrame) -> Tuple[pd.DataFrame, go.Figure, go.Figure]:

    df = df.copy()
    jfk = get_jfk(df)

    lat0 = float(jfk["lat"])
    lon0 = float(jfk["lon"])

    df["dist_euclid_deg"] = df.apply(lambda r: euclidean_distance_deg(lat0, lon0, float(r["lat"]), float(r["lon"])), axis=1)
    df["dist_geodesic_km"] = df.apply(lambda r: geodesic_distance_km(lat0, lon0, float(r["lat"]), float(r["lon"])), axis=1)

    # Basic stats
    stats = df[["dist_euclid_deg", "dist_geodesic_km"]].describe(percentiles=[0.25, 0.5, 0.75]).T
    stats = stats.rename(columns={"50%": "median"})

    fig_e = px.histogram(
        df,
        x="dist_euclid_deg",
        nbins=50,
        title="Distribution of Euclidean distances from JFK (in degree space)",
    )
    fig_e.update_layout(margin=dict(l=10, r=10, t=60, b=10))

    fig_g = px.histogram(
        df,
        x="dist_geodesic_km",
        nbins=50,
        title="Distribution of geodesic distances from JFK (km, R=6371)",
    )
    fig_g.update_layout(margin=dict(l=10, r=10, t=60, b=10))

    return stats, fig_e, fig_g


def analyze_timezones(df: pd.DataFrame) -> go.Figure:

    counts = (
        df.assign(tzone=df["tzone"].fillna("Unknown"))
          .groupby("tzone", as_index=False)
          .size()
          .sort_values("size", ascending=False)
    )

    fig = px.bar(
        counts,
        x="tzone",
        y="size",
        title="Time zones — number of airports per time zone (proxy for 'relative amount')",
    )
    fig.update_layout(xaxis_title="Time zone", yaxis_title="Number of airports", margin=dict(l=10, r=10, t=60, b=10))
    return fig



def main() -> None:
    # Load
    df = pd.read_csv(DATA_PATH)

    print("Loaded airports.csv")
    print(df.head())
    print(f"Rows: {len(df):,}  Columns: {list(df.columns)}")

    # Task 1
    fig_all = plot_all_airports_world(df, color_by_altitude=False)
    save_html(fig_all, "01_all_airports_world")

    # Extra color coded by altitude
    fig_all_alt = plot_all_airports_world(df, color_by_altitude=True)
    save_html(fig_all_alt, "01b_all_airports_world_altitude")

    # 2 outside US + US-only map
    fig_out, fig_us = plot_airports_us_and_outside(df, color_by_altitude=False)
    save_html(fig_out, "02_airports_outside_us")
    save_html(fig_us, "03_airports_us_only")

    # Extra altitude coloring
    fig_out_alt, fig_us_alt = plot_airports_us_and_outside(df, color_by_altitude=True)
    save_html(fig_out_alt, "02b_airports_outside_us_altitude")
    save_html(fig_us_alt, "03b_airports_us_only_altitude")

    # 3 function route map for one FAA code
    example_faa = "LAX"
    fig_route = plot_route_from_nyc(df, example_faa, us_only_if_us=True)
    save_html(fig_route, f"04_route_JFK_to_{example_faa}")

    #4 routes to multiple FAA codes (example)
    example_faas = ["LAX", "ORD", "MIA", "SEA"]
    fig_routes = plot_routes_from_nyc(df, example_faas, us_only_if_us=False)
    save_html(fig_routes, "05_routes_JFK_to_multiple")

    # 5 Euclidean distance distribution from JFK
    stats, fig_euclid, fig_geo = plot_distance_distributions(df)
    print("\nDistance summary statistics (from JFK):")
    print(stats)

    save_html(fig_euclid, "06_distribution_euclidean_degree_space")
    save_html(fig_geo, "07_distribution_geodesic_km")

    # 6 time zones analysis
    fig_tz = analyze_timezones(df)
    save_html(fig_tz, "08_timezones_airport_counts_proxy")

    print(f"\nSaved all interactive plots to: {OUT_DIR.resolve()}")


if __name__ == "__main__":
    main()import sqlite3
from contextlib import closing

with closing(sqlite3.connect("flights_database.db")) as con:
    df = pd.read_sql_query("SELECT * FROM flights LIMIT 5;", con)
    print(df)

In [ ]:
import sqlite3
import pandas as pd
import math
from pathlib import Path

# Distance function 
def geodesic_distance_km(lat1: float, lon1: float, lat2: float, lon2: float, R_km: float = 6371.0) -> float:
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    lam1 = math.radians(lon1)
    lam2 = math.radians(lon2)

    dphi = phi2 - phi1
    dlam = lam2 - lam1
    phi_m = 0.5 * (phi1 + phi2)

    term1 = 2.0 * math.sin(dphi / 2.0) * math.cos(dlam / 2.0)
    term2 = 2.0 * math.cos(phi_m) * math.sin(dlam / 2.0)

    return R_km * math.sqrt(term1 * term1 + term2 * term2)
# ---------------------------

DB_PATH = Path("flights_database.db")

with sqlite3.connect(DB_PATH) as conn:

    # Small sample (just to check DB works)
    df_sample = pd.read_sql_query(
        "SELECT * FROM flights LIMIT 5;",
        conn
    )
    print(df_sample)

    # Route-level distances from JFK
    sql = """
    SELECT 
        f.origin,
        f.dest,
        AVG(f.distance) AS flight_distance,
        a.lat,
        a.lon
    FROM flights f
    JOIN airports a
        ON f.dest = a.faa
    WHERE f.origin = 'JFK'
    GROUP BY f.origin, f.dest, a.lat, a.lon
    """
    
    df_routes = pd.read_sql_query(sql, conn)
    print(df_routes.head())

    # Get JFK coordinates
    jfk = pd.read_sql_query(
        "SELECT lat, lon FROM airports WHERE faa = 'JFK';",
        conn
    ).iloc[0]

# Connection automatically closes here

# Now continue calculations outside the DB block
lat0 = float(jfk["lat"])
lon0 = float(jfk["lon"])

df_routes["computed_distance_km"] = df_routes.apply(
    lambda r: geodesic_distance_km(
        lat0,
        lon0,
        float(r["lat"]),
        float(r["lon"])
    ),
    axis=1
)

KM_TO_MILES = 0.621371
df_routes["computed_distance_miles"] = (
    df_routes["computed_distance_km"] * KM_TO_MILES
)

df_routes["difference"] = (
    df_routes["flight_distance"] -
    df_routes["computed_distance_miles"]
)

print(df_routes[["dest", "flight_distance",
                 "computed_distance_miles",
                 "difference"]].head())

print("\nSummary of differences:")
print(df_routes["difference"].describe())

In [ ]:
# For many destinations the difference is tiny (ACK ~0.4 mi, ATL ~1.8 mi, BGR ~0.1 mi).
# Median difference ≈ 1.86 miles, that’s excellent.
# Mean difference is bigger (17 miles) because you have a few big outliers.
# Max difference ≈ 328.85 miles. that’s not normal.

In [ ]:
# extra analysis 

# worst mismatches
df_routes.sort_values("difference", ascending=False).head(10)[
    ["dest", "flight_distance", "computed_distance_miles", "difference", "lat", "lon"]
]

# the most negative 
df_routes.sort_values("difference", ascending=True).head(10)[
    ["dest", "flight_distance", "computed_distance_miles", "difference", "lat", "lon"]
]

In [ ]:
#visualization
fig = px.scatter(
    df_routes,
    x="computed_distance_miles",
    y="flight_distance",
    hover_name="dest",
    title="JFK route distances: computed (great-circle) vs flights.distance",
    labels={
        "computed_distance_miles": "Computed great-circle distance (miles)",
        "flight_distance": "Database flights.distance (miles)"
    }
)

min_val = min(df_routes["computed_distance_miles"].min(),
              df_routes["flight_distance"].min())

max_val = max(df_routes["computed_distance_miles"].max(),
              df_routes["flight_distance"].max())

fig.add_shape(
    type="line",
    x0=min_val,
    y0=min_val,
    x1=max_val,
    y1=max_val,
    line=dict(dash="dash")
)

fig.show()

In [ ]:
# data frame of NY airports
with sqlite3.connect(DB_PATH) as conn:
    df_nyc_airports = pd.read_sql_query(
        """
        SELECT *
        FROM airports
        WHERE faa IN (
            SELECT DISTINCT origin
            FROM flights
        );
        """,
        conn
    )

print(df_nyc_airports)

# three departure airports in NYC: EWR, JFK, and LGA

In [ ]:
import plotly.graph_objects as go
from typing import Sequence, Union, List

NYC_ORIGINS = {"JFK", "LGA", "EWR"}

def _route_lines_trace(origin_lat: float, origin_lon: float,
                       dest_lats: Sequence[float], dest_lons: Sequence[float]) -> go.Scattergeo:
    lats: List[Union[float, None]] = []
    lons: List[Union[float, None]] = []
    for lat, lon in zip(dest_lats, dest_lons):
        lats.extend([origin_lat, lat, None])
        lons.extend([origin_lon, lon, None])

    return go.Scattergeo(
        lat=lats,
        lon=lons,
        mode="lines",
        line=dict(width=2),
        hoverinfo="skip",
        name="Routes",
    )

def plot_destinations_on_day(month: int, day: int, nyc_airport: str,
                             db_path: Path = DB_PATH) -> go.Figure:
    """
    Plot all destinations flown to from a given NYC airport on a given (month, day) in 2023.
    Produces a Part-1 style route map with lines from origin to each destination.
    """
    origin = str(nyc_airport).upper().strip()
    if origin not in NYC_ORIGINS:
        raise ValueError(f"nyc_airport must be one of {sorted(NYC_ORIGINS)} (got {origin}).")

    with sqlite3.connect(db_path) as conn:
        # Origin coordinates
        df_origin = pd.read_sql_query(
            "SELECT faa, name, lat, lon FROM airports WHERE faa = ?;",
            conn,
            params=(origin,)
        )
        if df_origin.empty:
            raise ValueError(f"Origin airport {origin} not found in airports table.")
        origin_row = df_origin.iloc[0]
        origin_lat = float(origin_row["lat"])
        origin_lon = float(origin_row["lon"])

        # Distinct destinations for that day + origin, with destination coordinates
        sql = """
        SELECT DISTINCT f.dest, a.name, a.lat, a.lon
        FROM flights f
        JOIN airports a ON a.faa = f.dest
        WHERE f.year = 2023
          AND f.month = ?
          AND f.day = ?
          AND f.origin = ?
          AND a.lat IS NOT NULL
          AND a.lon IS NOT NULL
        ORDER BY f.dest;
        """
        df_dest = pd.read_sql_query(sql, conn, params=(month, day, origin))

    if df_dest.empty:
        raise ValueError(f"No destinations found for {origin} on 2023-{month:02d}-{day:02d}.")

    # Base map: plot destination points
    fig = px.scatter_geo(
        df_dest,
        lat="lat",
        lon="lon",
        hover_name="dest",
        hover_data={"name": True, "lat": False, "lon": False},
        title=f"Destinations from {origin} on 2023-{month:02d}-{day:02d} (n={len(df_dest)})",
    )
    fig.update_geos(showland=True)

    # Add route lines
    fig.add_trace(_route_lines_trace(origin_lat, origin_lon, df_dest["lat"].astype(float), df_dest["lon"].astype(float)))

    # Add origin marker
    fig.add_trace(
        go.Scattergeo(
            lat=[origin_lat],
            lon=[origin_lon],
            mode="markers",
            marker=dict(size=10),
            name=origin,
            text=[f"{origin}: {origin_row['name']}"],
            hoverinfo="text",
        )
    )

    fig.update_layout(margin=dict(l=10, r=10, t=60, b=10))
    return fig

# visualization

fig = plot_destinations_on_day(1, 1, "JFK")
fig.show()

In [ ]:
from typing import Dict, Any

DB_PATH = Path("flights_database.db")
NYC_ORIGINS = {"JFK", "LGA", "EWR"}

def day_flight_stats(month: int, day: int, nyc_airport: str, db_path: Path = DB_PATH) -> pd.DataFrame:
    origin = str(nyc_airport).upper().strip()
    if origin not in NYC_ORIGINS:
        raise ValueError(f"nyc_airport must be one of {sorted(NYC_ORIGINS)} (got {origin}).")

    with sqlite3.connect(db_path) as conn:
        # Basic counts + delay summaries
        sql_basic = """
        SELECT
            COUNT(*) AS n_flights,
            COUNT(DISTINCT dest) AS n_unique_destinations,
            COUNT(DISTINCT carrier) AS n_unique_carriers,

            AVG(dep_delay) AS avg_dep_delay,
            AVG(arr_delay) AS avg_arr_delay,

            -- SQLite has no MEDIAN built-in; compute via sorting + LIMIT/OFFSET below
            SUM(CASE WHEN dep_time IS NULL OR arr_time IS NULL THEN 1 ELSE 0 END) AS n_missing_times
        FROM flights
        WHERE year = 2023
          AND month = ?
          AND day = ?
          AND origin = ?;
        """
        basic = pd.read_sql_query(sql_basic, conn, params=(month, day, origin)).iloc[0].to_dict()

        # Most visited destination
        sql_top_dest = """
        SELECT dest, COUNT(*) AS n
        FROM flights
        WHERE year = 2023
          AND month = ?
          AND day = ?
          AND origin = ?
        GROUP BY dest
        ORDER BY n DESC, dest ASC
        LIMIT 1;
        """
        top = pd.read_sql_query(sql_top_dest, conn, params=(month, day, origin))
        if top.empty:
            raise ValueError(f"No flights found for {origin} on 2023-{month:02d}-{day:02d}.")
        top_dest = top.iloc[0]["dest"]
        top_dest_n = int(top.iloc[0]["n"])

        # Median dep_delay (SQLite trick: sort and pick middle)
        # Note: ignores NULL delays
        sql_median_dep = """
        WITH ordered AS (
          SELECT dep_delay
          FROM flights
          WHERE year = 2023 AND month = ? AND day = ? AND origin = ?
            AND dep_delay IS NOT NULL
          ORDER BY dep_delay
        ),
        cnt AS (SELECT COUNT(*) AS n FROM ordered)
        SELECT dep_delay AS median_dep_delay
        FROM ordered
        LIMIT 1
        OFFSET (SELECT (n - 1) / 2 FROM cnt);
        """
        med_dep = pd.read_sql_query(sql_median_dep, conn, params=(month, day, origin))
        median_dep_delay = float(med_dep.iloc[0]["median_dep_delay"]) if not med_dep.empty else float("nan")

        # Median arr_delay (same trick)
        sql_median_arr = """
        WITH ordered AS (
          SELECT arr_delay
          FROM flights
          WHERE year = 2023 AND month = ? AND day = ? AND origin = ?
            AND arr_delay IS NOT NULL
          ORDER BY arr_delay
        ),
        cnt AS (SELECT COUNT(*) AS n FROM ordered)
        SELECT arr_delay AS median_arr_delay
        FROM ordered
        LIMIT 1
        OFFSET (SELECT (n - 1) / 2 FROM cnt);
        """
        med_arr = pd.read_sql_query(sql_median_arr, conn, params=(month, day, origin))
        median_arr_delay = float(med_arr.iloc[0]["median_arr_delay"]) if not med_arr.empty else float("nan")

    n_flights = int(basic["n_flights"])
    n_missing_times = int(basic["n_missing_times"])
    cancelled_rate = (n_missing_times / n_flights) if n_flights > 0 else float("nan")

    out: Dict[str, Any] = {
        "date": f"2023-{month:02d}-{day:02d}",
        "origin": origin,
        "n_flights": n_flights,
        "n_unique_destinations": int(basic["n_unique_destinations"]),
        "n_unique_carriers": int(basic["n_unique_carriers"]),
        "top_destination": top_dest,
        "top_destination_flights": top_dest_n,
        "avg_dep_delay_min": float(basic["avg_dep_delay"]) if basic["avg_dep_delay"] is not None else float("nan"),
        "median_dep_delay_min": median_dep_delay,
        "avg_arr_delay_min": float(basic["avg_arr_delay"]) if basic["avg_arr_delay"] is not None else float("nan"),
        "median_arr_delay_min": median_arr_delay,
        "n_missing_dep_or_arr_time": n_missing_times,
        "missing_time_rate": cancelled_rate,
    }

    return pd.DataFrame([out])

# visualize it 

stats_df = day_flight_stats(1, 1, "JFK")
print(stats_df)

# extra: breakout tabel( top 10 destinations)

with sqlite3.connect(DB_PATH) as conn:
    top10 = pd.read_sql_query(
        """
        SELECT dest, COUNT(*) AS n_flights
        FROM flights
        WHERE year=2023 AND month=? AND day=? AND origin=?
        GROUP BY dest
        ORDER BY n_flights DESC
        LIMIT 10;
        """,
        conn,
        params=(1, 1, "JFK")
    )
print(top10)

In [ ]:
# 267 flights departed JFK on 2023-01-01

# They went to 64 unique destinations

# Operated by 7 carriers

# Most frequent destination: LAX with 18 flights

# Average departure delay: ~18.76 min

# Median departure delay: -1 min (half the flights left 1 minute early or less)

# Average arrival delay: ~7.49 min

# Median arrival delay: -7 min (typical flight arrived early)

# Missing dep/arr time: 1 flight → missing rate 0.37%

In [ ]:
def plane_type_usage(origin: str, destination: str, db_path: Path = DB_PATH) -> dict:
    """
    Returns a dictionary describing how many times each plane type
    was used for a given flight trajectory (origin -> destination).
    """

    origin = origin.upper().strip()
    destination = destination.upper().strip()

    with sqlite3.connect(db_path) as conn:
        sql = """
        SELECT p.type, COUNT(*) AS n_flights
        FROM flights f
        JOIN planes p
            ON f.tailnum = p.tailnum
        WHERE f.origin = ?
          AND f.dest = ?
          AND p.type IS NOT NULL
        GROUP BY p.type
        ORDER BY n_flights DESC;
        """

        df = pd.read_sql_query(sql, conn, params=(origin, destination))

    # Convert to dictionary
    return dict(zip(df["type"], df["n_flights"]))

usage = plane_type_usage("JFK", "LAX")
print(usage)

In [ ]:
def plot_plane_model_usage(origin: str, destination: str):
    origin = origin.upper().strip()
    destination = destination.upper().strip()

    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query(
            """
            SELECT p.model, COUNT(*) AS n_flights
            FROM flights f
            JOIN planes p
                ON f.tailnum = p.tailnum
            WHERE f.origin = ?
              AND f.dest = ?
              AND p.model IS NOT NULL
            GROUP BY p.model
            ORDER BY n_flights DESC;
            """,
            conn,
            params=(origin, destination)
        )

    if df.empty:
        raise ValueError("No aircraft model data found.")

    fig = px.bar(
        df,
        x="model",
        y="n_flights",
        title=f"Aircraft Model Usage: {origin} → {destination}",
        text="n_flights"
    )

    fig.update_layout(
        xaxis_title="Aircraft Model",
        yaxis_title="Number of Flights",
        margin=dict(l=20, r=20, t=60, b=20)
    )

    return fig

fig = plot_plane_model_usage("JFK", "LAX")
fig.show()

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    df_airline_delay = pd.read_sql_query(
        """
        SELECT 
            a.name AS airline_name,
            AVG(f.dep_delay) AS avg_dep_delay
        FROM flights f
        JOIN airlines a
            ON f.carrier = a.carrier
        WHERE f.dep_delay IS NOT NULL
        GROUP BY a.name
        ORDER BY avg_dep_delay DESC;
        """,
        conn
    )

print(df_airline_delay)

fig = px.bar(
    df_airline_delay,
    x="airline_name",
    y="avg_dep_delay",
    title="Average Departure Delay per Airline (2023)",
    labels={
        "airline_name": "Airline",
        "avg_dep_delay": "Average Departure Delay (minutes)"
    },
    text="avg_dep_delay"
)

fig.update_layout(
    xaxis_tickangle=45,  # rotate airline names
    margin=dict(l=20, r=20, t=60, b=120)
)

fig.show()

In [ ]:
def count_delayed_flights(start_month: int, end_month: int, destination: str,
                          db_path: Path = DB_PATH) -> int:
    """
    Returns the number of delayed flights (dep_delay > 0)
    to a given destination within a range of months (inclusive).
    """

    if start_month > end_month:
        raise ValueError("start_month must be <= end_month")

    destination = destination.upper().strip()

    with sqlite3.connect(db_path) as conn:
        result = conn.execute(
            """
            SELECT COUNT(*)
            FROM flights
            WHERE month BETWEEN ? AND ?
              AND dest = ?
              AND dep_delay > 0;
            """,
            (start_month, end_month, destination)
        ).fetchone()

    return result[0]

n = count_delayed_flights(1, 3, "LAX")
print("Number of delayed flights:", n)

In [ ]:
def top5_manufacturers_to_destination(dest: str, db_path: Path = DB_PATH) -> pd.DataFrame:
    """
    Returns a DataFrame with the top 5 airplane manufacturers (and flight counts)
    for flights whose destination is `dest`.
    """
    dest = str(dest).upper().strip()

    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query(
            """
            SELECT
                p.manufacturer,
                COUNT(*) AS n_flights
            FROM flights f
            JOIN planes p
              ON f.tailnum = p.tailnum
            WHERE f.dest = ?
              AND p.manufacturer IS NOT NULL
            GROUP BY p.manufacturer
            ORDER BY n_flights DESC
            LIMIT 5;
            """,
            conn,
            params=(dest,)
        )

    return df

print(top5_manufacturers_to_destination("LAX"))

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    df_delay = pd.read_sql_query(
        """
        SELECT distance, arr_delay
        FROM flights
        WHERE arr_delay IS NOT NULL
          AND distance IS NOT NULL;
        """,
        conn
    )

print(df_delay.head())

correlation = df_delay["distance"].corr(df_delay["arr_delay"])
print("Correlation between distance and arrival delay:", correlation)

fig = px.scatter(
    df_delay,
    x="distance",
    y="arr_delay",
    opacity=0.3,
    title="Relationship Between Flight Distance and Arrival Delay",
    labels={
        "distance": "Flight Distance (miles)",
        "arr_delay": "Arrival Delay (minutes)"
    }
)

fig.show()

fig = px.scatter(
    df_delay,
    x="distance",
    y="arr_delay",
    opacity=0.3,
    trendline="ols",
    title="Distance vs Arrival Delay (with Trendline)",
    labels={
        "distance": "Flight Distance (miles)",
        "arr_delay": "Arrival Delay (minutes)"
    }
)

fig.show()

In [ ]:
def fill_plane_speeds(db_path: Path = DB_PATH) -> None:
    """
    Groups flights by plane model (via tailnum->planes) and computes avg speed per model:
        avg_speed_mph = AVG(60 * distance / air_time)
    Then writes those speeds into planes.speed for all rows sharing that model.
    """
    with sqlite3.connect(db_path) as conn:
        cur = conn.cursor()

        # (Optional but helpful) ensure the speed column exists
        # If your assignment DB already has it, this does nothing.
        cur.execute("PRAGMA table_info(planes);")
        cols = {row[1] for row in cur.fetchall()}
        if "speed" not in cols:
            cur.execute("ALTER TABLE planes ADD COLUMN speed REAL;")

        # Build a temporary table of model -> avg_speed_mph
        cur.execute("DROP TABLE IF EXISTS model_speed;")
        cur.execute(
            """
            CREATE TEMP TABLE model_speed AS
            SELECT
                p.model AS model,
                AVG(60.0 * f.distance / f.air_time) AS avg_speed_mph
            FROM flights f
            JOIN planes p
              ON f.tailnum = p.tailnum
            WHERE p.model IS NOT NULL
              AND f.distance IS NOT NULL
              AND f.air_time IS NOT NULL
              AND f.air_time > 0
            GROUP BY p.model;
            """
        )

        # Update planes.speed using the model mapping
        cur.execute(
            """
            UPDATE planes
            SET speed = (
                SELECT ms.avg_speed_mph
                FROM model_speed ms
                WHERE ms.model = planes.model
            )
            WHERE model IS NOT NULL;
            """
        )

        # Commit changes
        conn.commit()

        # Print quick summary: how many planes got a speed value
        n_updated = cur.execute(
            "SELECT COUNT(*) FROM planes WHERE speed IS NOT NULL;"
        ).fetchone()[0]
        print(f"Filled planes.speed for {n_updated} plane records (speed not NULL).")

        # (Optional) peek at a few examples
        sample = cur.execute(
            """
            SELECT tailnum, manufacturer, model, speed
            FROM planes
            WHERE speed IS NOT NULL
            ORDER BY speed DESC
            LIMIT 5;
            """
        ).fetchall()
        print("Top 5 speeds (sample):")
        for row in sample:
            print(row)

fill_plane_speeds(DB_PATH)

In [ ]:
import sqlite3
import pandas as pd
import plotly.express as px

with sqlite3.connect(DB_PATH) as conn:
    df_speeds = pd.read_sql_query(
        """
        SELECT manufacturer, model, speed
        FROM planes
        WHERE speed IS NOT NULL;
        """,
        conn
    )

print(df_speeds["speed"].describe())

fig = px.histogram(
    df_speeds,
    x="speed",
    nbins=40,
    title="Distribution of Average Aircraft Speeds (mph)",
    labels={"speed": "Average Speed (mph)"}
)

fig.show()

df_speeds.sort_values("speed", ascending=False).head(10)

df_speeds.sort_values("speed").head(10)

df_speeds_clean = df_speeds[
    (df_speeds["speed"] > 200) &
    (df_speeds["speed"] < 700)
]

In [ ]:
import math

def initial_bearing(lat1, lon1, lat2, lon2):
    """
    Computes the initial bearing (degrees) from (lat1, lon1)
    to (lat2, lon2).
    """
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    d_lambda = math.radians(lon2 - lon1)

    x = math.sin(d_lambda) * math.cos(phi2)
    y = (math.cos(phi1) * math.sin(phi2) -
         math.sin(phi1) * math.cos(phi2) * math.cos(d_lambda))

    theta = math.atan2(x, y)

    bearing = math.degrees(theta)
    return (bearing + 360) % 360  # normalize

with sqlite3.connect(DB_PATH) as conn:
    jfk = pd.read_sql_query(
        "SELECT lat, lon FROM airports WHERE faa = 'JFK';",
        conn
    ).iloc[0]

lat0 = float(jfk["lat"])
lon0 = float(jfk["lon"])

with sqlite3.connect(DB_PATH) as conn:
    df_dest = pd.read_sql_query(
        """
        SELECT DISTINCT a.faa, a.name, a.lat, a.lon
        FROM flights f
        JOIN airports a ON f.dest = a.faa
        WHERE f.origin = 'JFK'
          AND a.lat IS NOT NULL
          AND a.lon IS NOT NULL;
        """,
        conn
    )

df_dest["direction_deg"] = df_dest.apply(
    lambda r: initial_bearing(lat0, lon0, float(r["lat"]), float(r["lon"])),
    axis=1
)

print(df_dest[["faa", "direction_deg"]].head())



In [ ]:
import numpy as np
import plotly.express as px

def plot_direction_polar_hist(df_dest: pd.DataFrame, origin_code: str, bin_size_deg: int = 10):
    """
    Polar bar chart of direction distribution.
    df_dest must have a column 'direction_deg' (0..360).
    """
    if "direction_deg" not in df_dest.columns:
        raise ValueError("df_dest must contain a 'direction_deg' column.")

    # Bin directions (e.g., 10-degree bins)
    bins = np.arange(0, 360 + bin_size_deg, bin_size_deg)
    df = df_dest.copy()
    df["dir_bin"] = pd.cut(df["direction_deg"], bins=bins, right=False, include_lowest=True)

    # Convert bin to its center angle for plotting
    df_counts = (
        df.groupby("dir_bin", as_index=False)
          .size()
          .rename(columns={"size": "n_destinations"})
    )
    df_counts["theta"] = df_counts["dir_bin"].apply(lambda x: (x.left + x.right) / 2)

    fig = px.bar_polar(
        df_counts,
        r="n_destinations",
        theta="theta",
        title=f"Direction of flights from {origin_code} to destinations (binned {bin_size_deg}°)",
        labels={"theta": "Direction (deg)", "n_destinations": "Number of destinations"},
    )
    fig.update_layout(margin=dict(l=20, r=20, t=60, b=20))
    return fig

fig = plot_direction_polar_hist(df_dest, origin_code="JFK", bin_size_deg=10)
fig.show()

def plot_direction_polar_scatter(df_dest: pd.DataFrame, origin_code: str):
    """
    Polar scatter: each destination plotted by direction and distance.
    df_dest must contain 'direction_deg' and 'distance_miles' columns.
    """
    needed = {"direction_deg", "distance_miles"}
    if not needed.issubset(df_dest.columns):
        raise ValueError(f"df_dest must contain columns: {sorted(needed)}")

    fig = px.scatter_polar(
        df_dest,
        r="distance_miles",
        theta="direction_deg",
        hover_name="faa" if "faa" in df_dest.columns else None,
        hover_data={"name": True} if "name" in df_dest.columns else None,
        title=f"Destinations from {origin_code}: direction vs distance",
        labels={"direction_deg": "Direction (deg)", "distance_miles": "Distance (miles)"},
    )
    fig.update_layout(margin=dict(l=20, r=20, t=60, b=20))
    return fig

KM_TO_MILES = 0.621371

df_dest["distance_miles"] = df_dest.apply(
    lambda r: geodesic_distance_km(lat0, lon0, float(r["lat"]), float(r["lon"])) * KM_TO_MILES,
    axis=1
)

fig2 = plot_direction_polar_scatter(df_dest, origin_code="JFK")
fig2.show()

